# Hybrid Architectures: Building Mixed Models

Build a hybrid SSM/attention model step by step using
`block_configs`. Combines Mamba-2
([Dao & Gu, 2024](https://arxiv.org/abs/2405.21060)) SSM layers
with GQA ([Ainslie et al., 2023](https://arxiv.org/abs/2305.13245))
attention layers, following patterns from Falcon-H1
([Zuo et al., 2025](https://arxiv.org/abs/2507.22448)) and Jamba
([Lieber et al., 2024](https://arxiv.org/abs/2403.19887)).

**Prerequisites:** `uv sync --extra dev --extra analysis`

In [1]:
import mlx.core as mx

from lmxlab.core.config import BlockConfig, ModelConfig
from lmxlab.models.base import LanguageModel

/Users/michaelellis/Projects/lmt-metal/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Per-Layer Block Configs

lmxlab's `ModelConfig` supports `block_configs`: a tuple of
`BlockConfig` objects, one per layer. This lets you mix
attention and SSM layers freely, similar to the hybrid designs
in Falcon-H1 and Jamba.

In [2]:
# Define an attention block config
attn_block = BlockConfig(
    attention="gqa",
    ffn="gated",
    norm="rms_norm",
    position="rope",
    d_model=64,
    n_heads=4,
    n_kv_heads=2,
    d_ff=128,
    bias=False,
    max_seq_len=128,
)

# Define a Mamba-2 SSM block config
ssm_block = BlockConfig(
    attention="mamba2",
    ffn="gated",
    norm="rms_norm",
    position="none",  # SSM layers don't use position encoding
    d_model=64,
    n_heads=4,
    n_kv_heads=2,
    d_ff=128,
    bias=False,
    max_seq_len=128,
    mamba_n_heads=4,
    mamba_head_dim=32,
    ssm_state_size=64,
    mamba_expand=2,
)

print("Attention block:")
print(f"  attention={attn_block.attention}, position={attn_block.position}")
print(f"\nSSM block:")
print(f"  attention={ssm_block.attention}, position={ssm_block.position}")
print(f"  mamba_n_heads={ssm_block.mamba_n_heads}, mamba_head_dim={ssm_block.mamba_head_dim}")

Attention block:
  attention=gqa, position=rope

SSM block:
  attention=mamba2, position=none
  mamba_n_heads=4, mamba_head_dim=32


## Build a Custom Hybrid Model

Let's create a 6-layer hybrid with pattern `MMMAMM`:
mostly SSM with one attention layer in the middle.

In [3]:
pattern = "MMMAMM"  # M=Mamba, A=Attention

blocks = tuple(
    attn_block if c == "A" else ssm_block
    for c in pattern
)

hybrid_config = ModelConfig(
    block=attn_block,  # Default block (used for final norm etc.)
    vocab_size=256,
    n_layers=len(pattern),
    block_configs=blocks,
    tie_embeddings=True,
)

print(f"Hybrid model: {len(pattern)} layers")
for i, c in enumerate(pattern):
    layer_type = "Attention (GQA)" if c == "A" else "SSM (Mamba-2)"
    print(f"  Layer {i}: {layer_type}")

Hybrid model: 6 layers
  Layer 0: SSM (Mamba-2)
  Layer 1: SSM (Mamba-2)
  Layer 2: SSM (Mamba-2)
  Layer 3: Attention (GQA)
  Layer 4: SSM (Mamba-2)
  Layer 5: SSM (Mamba-2)


In [4]:
hybrid = LanguageModel(hybrid_config)
mx.eval(hybrid.parameters())

print(f"Total params: {hybrid.count_parameters():,}")

tokens = mx.zeros((1, 16), dtype=mx.int32)
logits, caches = hybrid(tokens)
mx.eval(logits)
print(f"Output: {logits.shape}")

Total params: 349,180


Output: (1, 16, 256)


## Compare with Pre-Built Hybrids

lmxlab provides factory functions for published hybrid designs.

In [5]:
from lmxlab.models.falcon import falcon_h1_tiny
from lmxlab.models.jamba import jamba_tiny
from lmxlab.models.bamba import bamba_tiny

for name, factory in [("Falcon-H1", falcon_h1_tiny), ("Jamba", jamba_tiny), ("Bamba", bamba_tiny)]:
    cfg = factory()
    m = LanguageModel(cfg)
    mx.eval(m.parameters())
    n_ssm = sum(1 for i in range(cfg.n_layers) if "mamba" in cfg.get_block_config(i).attention)
    n_attn = cfg.n_layers - n_ssm
    print(f"{name}: {m.count_parameters():,} params, {n_ssm} SSM + {n_attn} attention layers")

Falcon-H1: 227,396 params, 3 SSM + 1 attention layers
Jamba: 495,944 params, 6 SSM + 2 attention layers
Bamba: 227,396 params, 3 SSM + 1 attention layers


## Activation Comparison: SSM vs Attention Layers

Use ActivationCapture to compare how SSM and attention
layers transform the residual stream.

In [6]:
from lmxlab.analysis.activations import ActivationCapture

with ActivationCapture(hybrid) as cap:
    hybrid(tokens)

norms = cap.layer_norms()

print(f"{'Layer':>5} {'Type':>6} {'Input':>8} {'Output':>8}")
print("-" * 30)
for i, c in enumerate(pattern):
    ltype = "Attn" if c == "A" else "SSM"
    in_norm = norms.get(f"layer_{i}/input", 0)
    out_norm = norms.get(f"layer_{i}/output", 0)
    print(f"{i:>5} {ltype:>6} {in_norm:>8.3f} {out_norm:>8.3f}")

Layer   Type    Input   Output
------------------------------
    0    SSM    3.670   14.878
    1    SSM   14.878   17.737
    2    SSM   17.737   21.093
    3   Attn   21.093   21.311
    4    SSM   21.311   25.849
    5    SSM   25.849   29.575


## References

- Vaswani et al. (2017). [Attention Is All You Need](https://arxiv.org/abs/1706.03762). NeurIPS 2017.
- Ainslie et al. (2023). [GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints](https://arxiv.org/abs/2305.13245). EMNLP 2023.
- Gu & Dao (2023). [Mamba: Linear-Time Sequence Modeling with Selective State Spaces](https://arxiv.org/abs/2312.00752). COLM 2024.
- Dao & Gu (2024). [Transformers are SSMs: Generalized Models and Efficient Algorithms Through Structured State Space Duality](https://arxiv.org/abs/2405.21060). ICML 2024.
- Lieber et al. (2024). [Jamba: A Hybrid Transformer-Mamba Language Model](https://arxiv.org/abs/2403.19887). arXiv preprint.
- IBM et al. (2025). [Bamba: Inference-Efficient Hybrid Mamba2 Model](https://huggingface.co/blog/bamba). HuggingFace blog post.
- Zuo et al. (2025). [Falcon-H1: A Family of Hybrid-Head Language Models Redefining Efficiency and Performance](https://arxiv.org/abs/2507.22448). arXiv preprint.